# Dziedziczenie a obiekty wartości — prostokąt i kwadrat

Z matematyki wiemy, że każdy kwadrat jest prostokątem. Czy klasa `Kwadrat` powinna wobec tego dziedziczyć po klasie `Prostokat`? Odpowiedź zależy również od operacji, które udostępniają te klasy.

To materiał dodatkowy do dziedziczenia. Wracamy w nim do rozróżnienia [obiektów wartości i obiektów, których tożsamość ma znaczenie](../02-referencje-kopie/02-referencje-kopie-dodatkowe.ipynb). Będziemy stopniowo zmieniać model i sprawdzać, jakie obietnice spełniają jego operacje.

## Prostokąt o zmiennych wymiarach

Pierwsza wersja pozwala zmieniać osobno szerokość i wysokość. Przyjmijmy umowę: `zmien_wysokosc` zmienia wysokość i pozostawia szerokość bez zmian; `zmien_szerokosc` działa analogicznie.

W tej prostej wersji pola są publiczne, więc można również przypisywać im wartości bezpośrednio. Problem, który zaraz zobaczymy, występuje nawet wtedy, gdy program korzysta wyłącznie z metod zmieniających wymiary.

In [ ]:
class Prostokat:
    def __init__(self, szerokosc, wysokosc):
        self.szerokosc = szerokosc
        self.wysokosc = wysokosc

    def zmien_szerokosc(self, nowa_szerokosc):
        self.szerokosc = nowa_szerokosc

    def zmien_wysokosc(self, nowa_wysokosc):
        self.wysokosc = nowa_wysokosc

    def pole(self):
        return self.szerokosc * self.wysokosc

    def __repr__(self):
        return f"Prostokat({self.szerokosc}x{self.wysokosc})"

In [ ]:
prostokat = Prostokat(5, 2)

In [ ]:
prostokat.zmien_wysokosc(10)

In [ ]:
prostokat

In [ ]:
prostokat.pole()

## Pierwsza próba: kwadrat dziedziczy wszystkie operacje

W konstruktorze kwadratu ustawimy obydwa wymiary na tę samą wartość. Dodamy też wygodną metodę zmieniającą długość boku.

In [ ]:
class NaiwnyKwadrat(Prostokat):
    def __init__(self, bok):
        super().__init__(bok, bok)

    def zmien_dlugosc_boku(self, nowy_bok):
        self.szerokosc = nowy_bok
        self.wysokosc = nowy_bok

    def __repr__(self):
        return f"NaiwnyKwadrat({self.szerokosc}x{self.wysokosc})"

In [ ]:
kwadrat = NaiwnyKwadrat(5)

In [ ]:
kwadrat

In [ ]:
kwadrat.zmien_dlugosc_boku(7)

In [ ]:
kwadrat

Ta operacja zachowała równość boków. Obiekt odziedziczył jednak także pozostałe metody `Prostokat`.

In [ ]:
kwadrat.zmien_wysokosc(10)

In [ ]:
kwadrat

Otrzymaliśmy `NaiwnyKwadrat(7x10)`. Klasa ma w nazwie kwadrat, ale jej obiekt nie zachowuje podstawowej własności kwadratu: równości boków. Dziedziczenie metod może udostępnić operacje, które nie pasują do modelu klasy pochodnej.

## Druga próba: kwadrat pilnuje równości boków

Nadpiszemy obie odziedziczone metody. Każda z nich zmieni teraz obydwa wymiary.

In [ ]:
class Kwadrat(Prostokat):
    def __init__(self, bok):
        super().__init__(bok, bok)

    def zmien_szerokosc(self, nowy_bok):
        self.szerokosc = nowy_bok
        self.wysokosc = nowy_bok

    def zmien_wysokosc(self, nowy_bok):
        self.szerokosc = nowy_bok
        self.wysokosc = nowy_bok

    def __repr__(self):
        return f"Kwadrat({self.szerokosc}x{self.wysokosc})"

In [ ]:
kwadrat = Kwadrat(5)

In [ ]:
kwadrat.zmien_wysokosc(10)

In [ ]:
kwadrat

Boki nadal są równe. Czy taka klasa może zastępować prostokąt w programie, który korzysta z wcześniejszej umowy?

### Program korzystający z prostokąta

Wyobraźmy sobie zmianę wysokości obrazu. Program zna aktualną szerokość i zakłada, że zmiana wysokości jej nie zmieni. Po ustawieniu wysokości na `10` pole powinno wynieść `szerokosc_przed * 10`.

Ważne jest zapisanie szerokości **przed** operacją. Gdybyśmy odczytali ją dopiero po zmianie, sprawdzenie nie wykryłoby naruszenia tej obietnicy.

In [ ]:
def przetestuj_prostokat(prostokat):
    print(f"Otrzymałem: {prostokat}")
    szerokosc_przed = prostokat.szerokosc
    prostokat.zmien_wysokosc(10)
    oczekiwane_pole = szerokosc_przed * 10
    rzeczywiste_pole = prostokat.pole()
    print(f"Po zmianie wysokości na 10: {prostokat}")
    if rzeczywiste_pole == oczekiwane_pole:
        print("Wszystko działa zgodnie z oczekiwaniami!")
    else:
        print(f"BŁĄD! Oczekiwano pola {oczekiwane_pole}, otrzymano {rzeczywiste_pole}")

In [ ]:
zwykly_prostokat = Prostokat(5, 2)

In [ ]:
przetestuj_prostokat(zwykly_prostokat)

In [ ]:
moj_kwadrat = Kwadrat(5)

In [ ]:
przetestuj_prostokat(moj_kwadrat)

Dla kwadratu otrzymaliśmy pole `100`, chociaż program oczekiwał `50`. Nie wystąpił wyjątek. Problem polega na tym, że `Kwadrat.zmien_wysokosc` zmienia również szerokość, wbrew umowie klasy bazowej.

### Co wynika z tego dla dziedziczenia?

To przykład związany z **zasadą podstawienia Liskov**: kod korzystający z umowy klasy bazowej powinien zachowywać poprawność również wtedy, gdy dostanie obiekt klasy pochodnej.

Samo nadpisanie metody jest zwykłym mechanizmem dziedziczenia. Tutaj trudność wynika ze zmiany jej znaczenia. Liczba nadpisanych metod także nie rozstrzyga, czy projekt jest dobry — trzeba sprawdzić, czy zachowują umowę.

Nie wynika z tego zakaz tworzenia mutowalnych figur. W edytorze grafiki możemy potrzebować obiektu, który zachowuje tożsamość przy zmianie rozmiaru. Możemy wtedy zdefiniować prostokąt i kwadrat jako osobne klasy, korzystające ze wspólnego interfejsu, np. metody `pole`.

## Inna umowa: obiekt opisuje wartość

Możemy potraktować wymiary figury jako **wartość**. W takim modelu prostokąt o wymiarach `5 x 2` i prostokąt `5 x 10` reprezentują różne wartości. Operacja może obliczać nową figurę, pozostawiając wcześniejszy obiekt bez zmian.

Nie nazywamy jej teraz `zmien_wysokosc`, lecz `z_wysokoscia`: wynikiem będzie nowy prostokąt o podanej wysokości. Nie wymagamy, żeby wynik miał tę samą klasę co argument. Rozciągnięcie kwadratu może dać zwykły prostokąt.

Użyjemy znanych już właściwości tylko do odczytu. Zwykły kod korzystający z publicznego interfejsu odczytuje wymiary, ale nie przypisuje im nowych wartości. Pola z podkreśleniem pozostają szczegółem implementacji, którego nie należy zmieniać z zewnątrz.

In [ ]:
class ProstokatVO:
    def __init__(self, szerokosc, wysokosc):
        if szerokosc <= 0 or wysokosc <= 0:
            raise ValueError("Wymiary muszą być dodatnie.")
        self._szerokosc = szerokosc
        self._wysokosc = wysokosc

    @property
    def szerokosc(self):
        return self._szerokosc

    @property
    def wysokosc(self):
        return self._wysokosc

    def z_szerokoscia(self, nowa):
        return ProstokatVO(nowa, self.wysokosc)

    def z_wysokoscia(self, nowa):
        return ProstokatVO(self.szerokosc, nowa)

    def pole(self):
        return self.szerokosc * self.wysokosc

    def __eq__(self, inny):
        if not isinstance(inny, ProstokatVO):
            return NotImplemented
        return self.szerokosc == inny.szerokosc and self.wysokosc == inny.wysokosc

    def __repr__(self):
        return f"ProstokatVO({self.szerokosc}x{self.wysokosc})"

`KwadratVO` ustawia dwa równe wymiary. Nie musi nadpisywać metod tworzących wynik o innych wymiarach: odziedziczone metody obiecują zwrócić prostokąt i właśnie to robią.

In [ ]:
class KwadratVO(ProstokatVO):
    def __init__(self, bok):
        super().__init__(bok, bok)

    def __repr__(self):
        return f"KwadratVO({self.szerokosc}x{self.wysokosc})"

In [ ]:
moj_matematyczny_kwadrat = KwadratVO(5)

In [ ]:
moj_matematyczny_kwadrat

In [ ]:
nowa_figura = moj_matematyczny_kwadrat.z_wysokoscia(10)

In [ ]:
nowa_figura

In [ ]:
moj_matematyczny_kwadrat

In [ ]:
moj_matematyczny_kwadrat.pole(), nowa_figura.pole()

In [ ]:
type(nowa_figura), type(moj_matematyczny_kwadrat)

Stary kwadrat ma nadal boki długości `5`. Wynikiem operacji jest prostokąt `5 x 10`. Przypisanie `moj_matematyczny_kwadrat.wysokosc = 10` zgłosi `AttributeError`, ponieważ właściwość nie ma metody ustawiającej.

In [ ]:
moj_matematyczny_kwadrat.wysokosc = 10

### Wartość a tożsamość

Dwa obiekty mogą reprezentować tę samą wartość, mimo że są odrębnymi obiektami Pythona. W naszym modelu o wartości decydują dwie długości boków; nie przechowujemy położenia ani innych cech figury.

In [ ]:
pierwsza = KwadratVO(5)
druga = ProstokatVO(5, 5)

In [ ]:
pierwsza == druga

In [ ]:
pierwsza is druga

Każdy obiekt Pythona ma tożsamość, także niemutowalny. W obiekcie wartości zwykle nie jest ona tym, co chcemy porównywać. Niemutowalność ułatwia bezpieczne współdzielenie wartości, ale nie jest definicją braku tożsamości.

Wcześniejszy przykład tras pokazuje też, że decyzja o traktowaniu czegoś jako wartości nie narzuca jednej techniki implementacji w każdym programie. Tu wybraliśmy brak publicznych operacji zmieniających stan, aby jasno określić znaczenie operacji i zachować wcześniejsze wartości.

## O czym trzeba zdecydować?

Dziedziczenie powinno odpowiadać umowie używanej przez program. W szczególności trzeba ustalić:

- czy operacja zmienia argument, czy tworzy nowy obiekt;
- jakie własności wcześniejszego obiektu pozostają zachowane;
- jakie typy wyników są dopuszczalne.

Gdyby `z_wysokoscia` miała obowiązek zwracać obiekt tej samej klasy co oryginał, nasza implementacja znów nie spełniałaby tej umowy dla kwadratu. Samo wprowadzenie niemutowalności nie rozwiązuje każdej trudności z dziedziczeniem.

## Ćwiczenia sprawdzające

1. Dlaczego `isinstance(moj_kwadrat, Prostokat)` zwraca `True`, chociaż funkcja `przetestuj_prostokat` wykryła problem?
2. Co stałoby się ze sprawdzeniem, gdyby oczekiwane pole obliczało na podstawie szerokości odczytanej po zmianie wysokości? Sprawdź to w osobnej wersji funkcji.
3. Czy usunięcie metod `zmien_szerokosc` i `zmien_wysokosc` z pierwszej klasy wystarczyłoby do uniemożliwienia zmiany jej stanu?
4. Utwórz `KwadratVO(5)` i wywołaj dla niego `z_szerokoscia(8)`. Sprawdź wymiary i typ wyniku oraz wymiary wcześniejszego obiektu.
5. Zastanów się, czy w tym modelu `KwadratVO(5)` i `ProstokatVO(5, 5)` powinny porównywać się jako równe. Która metoda o tym decyduje? Czy muszą być tym samym obiektem?

[Przykładowe odpowiedzi](06-prostokat-kwadrat-odpowiedzi.md)

Dokumentacja: [klasy i dziedziczenie](https://docs.python.org/3/tutorial/classes.html#inheritance), [`property`](https://docs.python.org/3/library/functions.html#property).